# Terraform Associate (004) — Prompt 12: State Backends, Locking, and Remote State

> **Exam Objective 6**: Understand state backends, state locking, remote state, and managing drift.

## Sections
1. [Local Backend](#1-local-backend)
2. [The `backend` Block](#2-the-backend-block)
3. [State Locking](#3-state-locking)
4. [Remote Backend — S3 + DynamoDB](#4-remote-backend--s3--dynamodb)
5. [Managing Resource Drift](#5-managing-resource-drift)
6. [State Manipulation Commands](#6-state-manipulation-commands)
7. [Exam-Style Questions](#7-exam-style-questions)
8. [Real-World Scenarios](#8-real-world-scenarios)

---
## 1. Local Backend

The **local backend** is the default when no `backend` block is configured. Terraform writes state to a JSON file on disk in the working directory.

### Key Facts

| Property | Detail |
|---|---|
| **State file** | `terraform.tfstate` in the current working directory |
| **Backup file** | `terraform.tfstate.backup` — created automatically before each apply, contains the previous state |
| **Locking** | ❌ No — no lock mechanism, risky for concurrent access |
| **Sharing** | ❌ No — the file lives on one machine |
| **Encryption at rest** | ❌ No — plaintext JSON on disk |
| **Team use** | ❌ Not suitable — each team member would have a different local state |

### When Local Backend Is Acceptable
- Learning and personal experiments
- Single-developer projects with no collaboration requirement
- CI environments where a single pipeline runner manages state

> ⭐ **Exam Tip**: `terraform.tfstate.backup` is created automatically before every `terraform apply`. It holds the **previous** state so you can roll back if something goes wrong. It is NOT versioned history — only the most recent backup is kept.

In [ ]:
# No backend block = local backend (default)
# State is written to ./terraform.tfstate

terraform {
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
  }
}

# Equivalent explicit local backend (rarely needed)
terraform {
  backend "local" {
    path = "./terraform.tfstate"  # default path
  }
}

---
## 2. The `backend` Block

The `backend` block configures where Terraform stores its state. It is declared inside the `terraform {}` block.

### Syntax

```hcl
terraform {
  backend "<backend_type>" {
    # backend-specific arguments
  }
}
```

### Supported Backend Types

| Backend | Locking | Notes |
|---|---|---|
| `local` | ❌ | Default; file on disk |
| `s3` | ✅ (with DynamoDB) | Most common AWS backend; `encrypt = true` recommended |
| `gcs` | ✅ | Google Cloud Storage |
| `azurerm` | ✅ | Azure Blob Storage |
| `http` | Optional | Generic HTTP endpoint |
| `kubernetes` | ✅ | Stores state in a Kubernetes Secret |
| `remote` (HCP Terraform) | ✅ | Deprecated in favor of the `cloud` block |

### Critical Rules

| Rule | Detail |
|---|---|
| **No variable references** | Backend configuration is evaluated before variables — you cannot use `var.x`, `local.x`, or data sources in the `backend` block |
| **`terraform init` required** | Must re-run `terraform init` after any change to the backend block — Terraform will offer to migrate existing state |
| **One backend per workspace** | Only one `backend` block is allowed in a configuration |
| **Partial configuration** | Sensitive values (credentials) can be supplied interactively or via `-backend-config` flag instead of hardcoding |

> ⭐ **Exam Tip**: Backend configuration **cannot use variable references** because the backend is initialized before the variable system is loaded. This is a common exam trap. Use partial configuration (`-backend-config` flag or a `.tfbackend` file) to supply dynamic values.

In [ ]:
# After changing the backend block, terraform init must be re-run
terraform init
# Terraform detects backend change and prompts:
# "Do you want to copy existing state to the new backend? yes/no"

# Partial configuration — supply backend values via CLI flag
# (useful for sensitive credentials or environment-specific values)
terraform init -backend-config="bucket=my-tfstate-bucket" \
               -backend-config="key=app/prod/terraform.tfstate" \
               -backend-config="region=us-east-1"

# Or supply via a separate .tfbackend file
terraform init -backend-config=prod.tfbackend

---
## 3. State Locking

**State locking** prevents two operations from modifying state simultaneously, avoiding corruption.

### How It Works

1. When `terraform plan`, `terraform apply`, or `terraform destroy` begins, Terraform acquires a lock.
2. If another operation tries to run concurrently, it sees the lock and either waits (if the backend supports timeouts) or immediately errors.
3. When the operation completes (or fails), the lock is released.

### Backend Locking Support

| Backend | Locking Support | Mechanism |
|---|---|---|
| `local` | ❌ None | N/A |
| `s3` | ✅ Optional | DynamoDB table (`dynamodb_table` argument) |
| `gcs` | ✅ Built-in | GCS object locking |
| `azurerm` | ✅ Built-in | Azure Blob leasing |
| HCP Terraform | ✅ Built-in | Managed by HCP Terraform platform |
| `http` | ✅ Optional | If the server supports lock/unlock endpoints |

### Force-Unlock

If a `terraform apply` is interrupted (killed process, network failure), the lock may not be released. Use `terraform force-unlock` to release it manually.

```bash
terraform force-unlock <lock-id>
```

> ⚠️ **Caution**: Only run `force-unlock` when you are certain no other operation is actively running. Unlocking while an apply is in progress will cause state corruption.

> ⭐ **Exam Tip**: The S3 backend does NOT provide locking on its own. Locking requires a **separate DynamoDB table**. Without the `dynamodb_table` argument, S3 backend has no lock.

In [ ]:
# Force-unlock a stuck lock
# First, find the lock ID from the error message Terraform shows when it detects a lock
# Example lock ID: 5b5b7e9a-8d4a-4f3a-b1c2-0e9d7f6a2c1b

terraform force-unlock 5b5b7e9a-8d4a-4f3a-b1c2-0e9d7f6a2c1b

# Terraform will prompt for confirmation:
# "Do you really want to force-unlock?
#  There is no safety guarantee! Only unlock if you are sure no other apply is running."

# Disable locking for a single operation (use sparingly)
terraform apply -lock=false
terraform plan  -lock=false

# Set lock timeout (default: 0s = fail immediately if locked)
terraform apply -lock-timeout=5m

---
## 4. Remote Backend — S3 + DynamoDB

The **S3 backend** is the most common AWS-based remote backend and the most heavily tested pattern on the Terraform Associate exam.

### Architecture

```
┌─────────────────────────────┐
│  Terraform CLI / CI Runner  │
└────────────┬────────────────┘
             │
      backend "s3"
             │
    ┌────────┴────────┐
    │                 │
  S3 Bucket        DynamoDB Table
  (state file)     (lock record)
```

### Key Arguments for S3 Backend

| Argument | Required | Description |
|---|---|---|
| `bucket` | ✅ | Name of the S3 bucket |
| `key` | ✅ | Path/filename within the bucket (e.g., `app/prod/terraform.tfstate`) |
| `region` | ✅ | AWS region of the S3 bucket |
| `encrypt` | Recommended | `true` — enables server-side encryption of the state file |
| `dynamodb_table` | For locking | Name of the DynamoDB table for state locking |
| `profile` | Optional | AWS credentials profile to use |
| `role_arn` | Optional | IAM role to assume for access |

> ⭐ **Exam Tips**:
> - Always set `encrypt = true` for S3 backend in production.
> - The DynamoDB table must have a **primary key attribute named `LockID`** of type String.
> - Without `dynamodb_table`, the S3 backend has **no locking**.

In [ ]:
# backend.tf — S3 + DynamoDB backend configuration
terraform {
  backend "s3" {
    bucket         = "my-company-terraform-state"
    key            = "applications/web-app/prod/terraform.tfstate"
    region         = "us-east-1"
    encrypt        = true                  # encrypt state at rest
    dynamodb_table = "terraform-state-lock" # enables state locking
  }
}

# The DynamoDB table must be created separately (bootstrap)
# DynamoDB table requirements:
#   - Billing mode: PAY_PER_REQUEST or provisioned
#   - Hash key (partition key): LockID  (type: String)

# Bootstrapping the lock table before using this backend:
resource "aws_dynamodb_table" "terraform_locks" {
  name         = "terraform-state-lock"
  billing_mode = "PAY_PER_REQUEST"
  hash_key     = "LockID"              # required attribute name

  attribute {
    name = "LockID"
    type = "S"                         # String type
  }
}

resource "aws_s3_bucket" "terraform_state" {
  bucket = "my-company-terraform-state"
}

resource "aws_s3_bucket_versioning" "terraform_state" {
  bucket = aws_s3_bucket.terraform_state.id
  versioning_configuration {
    status = "Enabled"                 # recommended: keeps state history
  }
}

resource "aws_s3_bucket_server_side_encryption_configuration" "terraform_state" {
  bucket = aws_s3_bucket.terraform_state.id
  rule {
    apply_server_side_encryption_by_default {
      sse_algorithm = "AES256"
    }
  }
}

---
## 5. Managing Resource Drift

**Drift** occurs when real infrastructure diverges from what Terraform's state file records — typically caused by manual changes in the cloud console or by another tool.

### Drift Detection Workflow

```
Real Infrastructure ←── manual change ──→ now differs from state
        │
        ▼
terraform plan -refresh-only
        │
        ▼
  Shows what state update would look like (no resource changes)
        │
        ▼
terraform apply -refresh-only
        │
        ▼
  State file updated to match reality (resources NOT changed)
```

### Drift-Related Commands

| Command | What It Does |
|---|---|
| `terraform plan` | Refreshes state AND shows config diffs — use for full reconciliation |
| `terraform plan -refresh-only` | Shows only what would be updated in state to match reality — no config changes shown |
| `terraform apply -refresh-only` | Accepts drift into state without changing real resources |
| `terraform plan -refresh=false` | Skips state refresh — faster but may use stale data |

### Drift Scenarios

| Scenario | Recommended Action |
|---|---|
| Someone added a tag manually | `-refresh-only` to accept into state; or next `plan`/`apply` will remove it |
| Someone resized an instance manually | `-refresh-only` to accept; or next `apply` will resize back to declared value |
| Resource was deleted manually | `-refresh-only` or `state rm` to remove from state; Terraform will recreate on next apply |

> ⭐ **Exam Tip**: `terraform apply -refresh-only` updates the **state file** to match reality. It does **NOT** modify real infrastructure. The next regular `terraform apply` may then show changes to reconcile config with the refreshed state.

In [ ]:
# Scenario: Someone manually changed the instance type in the console
# State says: t3.medium  |  Reality: t3.large

# Step 1: Check drift without changing anything
terraform plan -refresh-only
# Output:
# ~ aws_instance.web
#   ~ instance_type = "t3.medium" -> "t3.large"  (refresh)
# Note: These changes only update the Terraform state.

# Step 2a: Accept drift (record the manual change in state)
terraform apply -refresh-only
# State now records t3.large; config still says t3.medium
# Next terraform apply will show a diff and offer to resize back to t3.medium

# Step 2b: Reject drift (re-apply config to restore declared state)
terraform apply
# Terraform will resize the instance back to t3.medium as declared in config

# Skip refresh for speed (use only when state is known to be current)
terraform plan -refresh=false

---
## 6. State Manipulation Commands

### Declarative Approach (Preferred — Terraform 1.1+)

#### `moved` Block

Use the `moved` block to update state when you rename a resource or move it into/out of a module — **without destroying and recreating** the real infrastructure.

```hcl
moved {
  from = <old_address>
  to   = <new_address>
}
```

#### `removed` Block (Terraform 1.7+)

Use the `removed` block to stop managing a resource without destroying the real infrastructure. Terraform removes it from state.

```hcl
removed {
  from = <resource_address>

  lifecycle {
    destroy = false  # do NOT destroy real infrastructure
  }
}
```

### Imperative `terraform state` Subcommands

| Command | Purpose |
|---|---|
| `terraform state list` | List all resource addresses tracked in state |
| `terraform state show <addr>` | Show all attributes of a specific resource |
| `terraform state mv <old> <new>` | Move/rename a resource in state (older approach; `moved` block preferred) |
| `terraform state rm <addr>` | Remove a resource from state without destroying it |
| `terraform state pull` | Download current remote state as JSON to stdout |
| `terraform state push <file>` | Upload a local state file to the remote backend |

> ⭐ **Exam Tip**: `terraform state rm` removes a resource from state but does **not** destroy the real cloud resource. Use it when you want Terraform to "forget" a resource (e.g., to adopt it under a different address, or to manage it another way).

> ⭐ **Exam Tip**: `terraform state push` is rarely used and potentially dangerous — pushing an incorrect state can cause Terraform to lose track of real resources. Use with extreme caution.

In [ ]:
# Example 1: moved block — rename a resource without recreating it
# Before: resource "aws_instance" "server" { ... }
# After:  resource "aws_instance" "web_server" { ... }  (renamed)

moved {
  from = aws_instance.server
  to   = aws_instance.web_server
}

# terraform plan will show:
#   # aws_instance.server has moved to aws_instance.web_server
#   resource "aws_instance" "web_server" {  (no change)

# Example 2: moved block — move a resource INTO a module
# Before (root): resource "aws_s3_bucket" "logs" { ... }
# After: moved into module "storage"

moved {
  from = aws_s3_bucket.logs
  to   = module.storage.aws_s3_bucket.logs
}

In [ ]:
# Example: removed block — stop managing a resource without destroying it
# Use case: a manually-managed S3 bucket that was temporarily imported;
# we want Terraform to stop tracking it without deleting the bucket.

removed {
  from = aws_s3_bucket.legacy_assets

  lifecycle {
    destroy = false  # false = do NOT destroy the real resource
                     # true  = destroy the real resource (same as normal removal)
  }
}

# After applying, aws_s3_bucket.legacy_assets is gone from state.
# The actual S3 bucket still exists in AWS — Terraform just no longer tracks it.

In [ ]:
# terraform state list — show all tracked resources
terraform state list
# Output example:
# aws_instance.web_server
# aws_security_group.web
# aws_vpc.main
# module.database.aws_db_instance.primary

# terraform state list with prefix filter
terraform state list module.database
# module.database.aws_db_instance.primary
# module.database.aws_db_subnet_group.main

# terraform state show — inspect a specific resource
terraform state show aws_instance.web_server
# Output: all attributes including computed values (id, arn, private_ip, etc.)

# terraform state mv — rename resource (imperative; moved block preferred)
terraform state mv aws_instance.server aws_instance.web_server

# terraform state rm — remove from state without destroying
terraform state rm aws_s3_bucket.legacy_assets

# terraform state pull — download state JSON
terraform state pull > state-backup.json

# terraform state push — upload state (DANGEROUS)
terraform state push state-backup.json

---
## 7. Exam-Style Questions

### Question 1

A team is using the S3 backend to store Terraform state. After a CI pipeline process was forcibly terminated, subsequent runs fail with: `Error: Error locking state: Error acquiring the state lock`. What is the correct approach to resolve this?

- A. Delete the `terraform.tfstate` file from the S3 bucket and re-run `terraform init`.
- B. Run `terraform force-unlock <lock-id>` after confirming no other operation is running.
- C. Run `terraform state push` to reset the lock.
- D. Re-run `terraform init -reconfigure` to clear the lock.

<details><summary>Answer</summary>

**B. Run `terraform force-unlock <lock-id>` after confirming no other operation is running.**

`terraform force-unlock` releases a stuck lock in the backend. The lock ID is shown in the error message Terraform displays when it detects the existing lock. This should only be done after confirming that the previous operation truly stopped — running force-unlock while an apply is still in progress risks state corruption. Options A, C, and D do not address the lock mechanism.

</details>

### Question 2

A developer changed the `backend` block from `local` to `s3`. When they run `terraform init`, what happens?

- A. Terraform automatically copies the local state to S3 without prompting.
- B. Terraform throws an error because backends cannot be changed after initialization.
- C. Terraform prompts the user to confirm whether to migrate existing state to the new backend.
- D. Terraform creates a new empty state in S3 and discards the local state.

<details><summary>Answer</summary>

**C. Terraform prompts the user to confirm whether to migrate existing state to the new backend.**

When `terraform init` detects a backend change, it offers to migrate state from the old backend to the new one. The user is explicitly prompted: `"Do you want to copy existing state to the new backend?"`. Answering `yes` copies the state. Answering `no` leaves the old state in place and initializes the new backend with no state. Terraform never discards state silently.

</details>

### Question 3

An operator discovers that a team member manually increased the size of an RDS database storage outside of Terraform. The operator wants Terraform's state to reflect the current actual size, without making any further changes to the real infrastructure. Which command achieves this?

- A. `terraform apply`
- B. `terraform refresh`
- C. `terraform apply -refresh-only`
- D. `terraform plan -refresh=false`

<details><summary>Answer</summary>

**C. `terraform apply -refresh-only`**

`terraform apply -refresh-only` reads the current state of all managed resources and updates the **state file** to match reality. It does NOT apply any changes to actual infrastructure. Option A (`terraform apply`) would detect the drift and then try to revert the storage size back to what is declared in configuration. Option B (`terraform refresh`) is a deprecated command that still works in older versions but is replaced by `-refresh-only` workflows. Option D skips the refresh entirely.

</details>

### Question 4

A Terraform configuration uses the following `backend` block:

```hcl
terraform {
  backend "s3" {
    bucket         = var.state_bucket   # uses a variable reference
    key            = "prod/terraform.tfstate"
    region         = "us-east-1"
  }
}
```

What happens when `terraform init` is run?

- A. Terraform evaluates the variable and uses its value for the bucket name.
- B. Terraform prompts for the value of `var.state_bucket` at init time.
- C. Terraform throws an error because variable references are not allowed in backend configuration.
- D. Terraform uses an empty string for `var.state_bucket` since variables are not yet loaded.

<details><summary>Answer</summary>

**C. Terraform throws an error because variable references are not allowed in backend configuration.**

Backend configuration is processed during the earliest phase of `terraform init`, before the variable system is initialized. Therefore, `var.x`, `local.x`, data sources, and resource attributes cannot be used inside a `backend` block. To supply dynamic values, use **partial configuration** — leave the backend block empty or incomplete, and pass values via `-backend-config` flags or a `.tfbackend` file at init time:

```bash
terraform init -backend-config="bucket=my-state-bucket"
```

</details>

---
## 8. Real-World Scenarios

<details>
<summary>Scenario 1: Bootstrapping an S3 + DynamoDB Remote Backend for a Team</summary>

**Situation**: A team is starting a new project and needs to set up a shared remote backend so that all engineers and CI pipelines use the same state file with locking. They will bootstrap the S3 bucket and DynamoDB table first, then migrate to the remote backend.

**Exam Sub-Objective**: S3 backend configuration; DynamoDB locking; `encrypt = true`; `dynamodb_table`; `terraform init` migration.

**Step 1 — Bootstrap resources (local backend)**:
```hcl
# bootstrap/main.tf — run with local backend to create the infrastructure
provider "aws" { region = "us-east-1" }

resource "aws_s3_bucket" "state" {
  bucket = "acme-corp-terraform-state"
}

resource "aws_s3_bucket_versioning" "state" {
  bucket = aws_s3_bucket.state.id
  versioning_configuration { status = "Enabled" }
}

resource "aws_s3_bucket_server_side_encryption_configuration" "state" {
  bucket = aws_s3_bucket.state.id
  rule {
    apply_server_side_encryption_by_default { sse_algorithm = "AES256" }
  }
}

resource "aws_dynamodb_table" "lock" {
  name         = "terraform-state-lock"
  billing_mode = "PAY_PER_REQUEST"
  hash_key     = "LockID"   # exact name required
  attribute {
    name = "LockID"
    type = "S"
  }
}
```

**Step 2 — Configure backend in the main project**:
```hcl
# main project: backend.tf
terraform {
  backend "s3" {
    bucket         = "acme-corp-terraform-state"
    key            = "projects/web-app/prod/terraform.tfstate"
    region         = "us-east-1"
    encrypt        = true
    dynamodb_table = "terraform-state-lock"
  }
}
```

**CLI Commands**:
```bash
cd bootstrap && terraform init && terraform apply  # create S3 + DynamoDB
cd ../main-project
terraform init  # Terraform migrates state to S3, confirms with prompt
```

**Expected Outcome**: All team members and pipelines now share the same state file. Concurrent runs are prevented by DynamoDB locking. State is encrypted at rest in S3.

</details>

<details>
<summary>Scenario 2: Detecting and Accepting Configuration Drift</summary>

**Situation**: A production EC2 instance was manually scaled up from `t3.medium` to `t3.large` by an on-call engineer during an incident. The Terraform configuration still declares `t3.medium`. The team needs to decide: accept the manual change permanently (update state) or revert back to the declared size.

**Exam Sub-Objective**: `terraform plan -refresh-only`; `terraform apply -refresh-only`; drift detection; reconciliation strategy.

**CLI Commands and Outputs**:
```bash
# Step 1: Detect drift — what changed outside Terraform?
terraform plan -refresh-only
# Output:
# Terraform will perform the following actions:
#   ~ aws_instance.web (refresh)
#     ~ instance_type = "t3.medium" -> "t3.large"
# Note: This plan does not change real infrastructure.
#       These changes only update the Terraform state.

# Step 2a: Accept the drift — record t3.large in state
terraform apply -refresh-only
# State now shows t3.large. Config still says t3.medium.
# Next terraform plan will propose changing back to t3.medium.

# Step 2b: Reject the drift — update config and reapply
# Edit main.tf: change instance_type to t3.large
# Then:
terraform apply  # no changes needed — config now matches reality

# OR: revert to t3.medium (reject the manual change)
terraform apply  # resizes back to t3.medium as declared
```

**Expected Outcome**: The team explicitly chooses whether to accept or reject the drift. Using `-refresh-only` gives them visibility and control rather than blindly reverting or accepting.

**Key Takeaway**: `apply -refresh-only` updates state to match reality. It does NOT touch real resources. The next regular apply may then show diffs between the refreshed state and the declared configuration.

</details>

<details>
<summary>Scenario 3: Renaming a Resource Without Destroying It Using `moved` Block</summary>

**Situation**: During a refactoring effort, a team renames several Terraform resources for clarity (e.g., `aws_instance.server` → `aws_instance.web_server`) and moves some resources into a module. Without a `moved` block, Terraform would destroy and recreate these resources, causing downtime.

**Exam Sub-Objective**: `moved` block; refactoring without destroy/recreate; state address updates.

**Before Refactoring**:
```hcl
resource "aws_instance" "server" {   # old name
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.medium"
}
```

**After Refactoring + moved block**:
```hcl
# moved.tf — declare all address changes
moved {
  from = aws_instance.server
  to   = aws_instance.web_server
}

# Also move a resource into a new module
moved {
  from = aws_security_group.app
  to   = module.compute.aws_security_group.app
}

# main.tf — updated resource name
resource "aws_instance" "web_server" {   # new name
  ami           = "ami-0c55b159cbfafe1f0"
  instance_type = "t3.medium"
}
```

**CLI Commands**:
```bash
terraform plan
# Output:
#   # aws_instance.server has moved to aws_instance.web_server
#     resource "aws_instance" "web_server" {   (no changes planned)
#   # aws_security_group.app has moved to module.compute.aws_security_group.app
#     resource "aws_security_group" "app" {    (no changes planned)
#
#   Plan: 0 to add, 0 to change, 0 to destroy.

terraform apply  # applies the address change in state; no infrastructure change
```

**Expected Outcome**: Resources are renamed in state without any infrastructure being destroyed or recreated. Zero downtime refactoring.

**Key Takeaway**: The `moved` block is the declarative, reviewable, and team-friendly way to rename resources. It shows up in `terraform plan` output for peer review. The older `terraform state mv` command achieves the same result imperatively but does not appear in plan output.

</details>

<details>
<summary>Scenario 4: Releasing a Stuck State Lock After CI Pipeline Failure</summary>

**Situation**: A CI/CD pipeline running `terraform apply` was killed mid-execution due to a timeout. Subsequent pipeline runs now fail immediately with a state lock error. The team needs to release the lock safely.

**Exam Sub-Objective**: `terraform force-unlock`; state locking; lock ID; safety precautions.

**Error Message (from failed run)**:
```
Error: Error locking state: Error acquiring the state lock: ConditionalCheckFailedException
  Lock Info:
    ID:        f23a8c4e-9d1b-4a7f-b3e2-0c8d5a6f1b2e
    Path:      my-company-terraform-state/projects/web-app/prod/terraform.tfstate
    Operation: OperationTypeApply
    Who:       runner@ci-server-01
    Version:   1.7.0
    Created:   2026-04-23 14:30:00 UTC
    Info:
```

**Resolution Steps**:
```bash
# Step 1: Confirm the original operation is truly dead
# Check CI logs, running processes on ci-server-01, and AWS console

# Step 2: Release the lock (use the ID from the error message)
terraform force-unlock f23a8c4e-9d1b-4a7f-b3e2-0c8d5a6f1b2e
# Terraform prompts:
# "Do you really want to force-unlock?
#  Terraform will remove the lock on the remote state.
#  This will allow local Terraform commands to modify this state, even though it
#  may be still be in use. Only 'yes' will be accepted to confirm."
#
# Enter a value: yes

# Step 3: Verify the state is consistent
terraform plan  # should run cleanly now

# Step 4: If state may be partially corrupted, check state integrity
terraform state pull | python -m json.tool > /dev/null && echo "State JSON is valid"
```

**Expected Outcome**: The DynamoDB lock record is deleted. Subsequent CI runs can acquire the lock and proceed normally.

**Key Takeaway**: The lock ID is always shown in the error message. `force-unlock` takes only the ID as an argument (not the full resource path). Never force-unlock without first verifying the original process has truly stopped.

</details>

<details>
<summary>Scenario 5: Adopting an Existing Resource with `removed` Block and `terraform state rm`</summary>

**Situation**: A legacy S3 bucket (`my-company-assets`) was previously managed by Terraform but was imported and adopted by a separate team using a different Terraform workspace. The current workspace still has the bucket in its state, causing conflicts. The team needs Terraform to stop tracking it without deleting the actual bucket.

**Exam Sub-Objective**: `terraform state rm`; `removed` block; stopping management without destroy; state hygiene.

**Approach A: Declarative `removed` block (Terraform 1.7+)**:
```hcl
# removed.tf — tell Terraform to stop tracking this resource
removed {
  from = aws_s3_bucket.company_assets

  lifecycle {
    destroy = false   # IMPORTANT: do not destroy the real bucket
  }
}

# Also remove the resource block from main.tf (delete it)
```

```bash
terraform plan
# Output:
#   # aws_s3_bucket.company_assets will no longer be managed by Terraform
#   # (destroy = false — resource will not be destroyed)
#   Plan: 0 to add, 0 to change, 0 to destroy, 1 to forget.

terraform apply  # removes from state; bucket remains in AWS
```

**Approach B: Imperative `terraform state rm` (older method)**:
```bash
# First, confirm the resource address
terraform state list | grep s3
# aws_s3_bucket.company_assets

# Remove from state without destroying
terraform state rm aws_s3_bucket.company_assets
# Removed aws_s3_bucket.company_assets from state

# Now delete the resource block from main.tf
# (otherwise next plan will try to create it)
```

**Expected Outcome**: The S3 bucket remains in AWS, intact and accessible. Terraform no longer tracks it in this workspace's state. The other team's workspace now has sole ownership.

**Key Takeaway**: `terraform state rm` and the `removed` block both remove a resource from Terraform's tracking without destroying real infrastructure. The `removed` block is preferred (1.7+) because it is declarative and visible in plan output. `terraform state rm` is immediate and does not require an apply cycle.

</details>

---
## Quick-Reference Cheat Sheet

### Backend Comparison

| Backend | Locking | Encryption | Team-Ready | Notes |
|---|---|---|---|---|
| `local` | ❌ | ❌ | ❌ | Default; fine for solo/learning |
| `s3` | ✅ (DynamoDB) | Optional (`encrypt=true`) | ✅ | Most tested on exam; needs DynamoDB for locking |
| `gcs` | ✅ | ✅ | ✅ | Google Cloud |
| `azurerm` | ✅ | ✅ | ✅ | Azure Blob |
| HCP Terraform | ✅ | ✅ | ✅ | Managed service; `cloud` block |

### Drift Management Commands

| Command | Effect |
|---|---|
| `terraform plan` | Refreshes state + shows config diffs |
| `terraform plan -refresh-only` | Shows what state update would look like (no resource changes) |
| `terraform apply -refresh-only` | Updates state to match reality; no resource changes |
| `terraform plan -refresh=false` | Skips refresh; uses cached state |

### State Manipulation: Declarative vs. Imperative

| Goal | Declarative (Preferred) | Imperative |
|---|---|---|
| Rename resource | `moved` block | `terraform state mv` |
| Stop managing without destroy | `removed { lifecycle { destroy = false } }` | `terraform state rm` |
| Download state | N/A | `terraform state pull` |
| Upload state | N/A | `terraform state push` |

### Most-Tested State Facts

1. **S3 backend needs DynamoDB** for locking — `dynamodb_table = "..."` + `LockID` primary key.
2. **`encrypt = true`** enables server-side encryption on the S3 state file.
3. **Backend block cannot use `var.x`** — evaluated before variables are loaded.
4. **`terraform init` required** after any backend change — offers state migration.
5. **`apply -refresh-only`** updates state only; does NOT change real resources.
6. **`terraform state rm`** removes from state; does NOT destroy real infrastructure.
7. **`terraform force-unlock`** releases stuck locks — use with caution.
8. **`terraform.tfstate.backup`** is the previous state, auto-created before each apply.